In [1]:
5

5

# Selenium

In [76]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time

# Setup driver
service = Service("D:\Adrta\chromedriver.exe")
driver = webdriver.Chrome(service=service)

driver.get("https://books.toscrape.com/")

# Small wait (page is static, but good habit)
time.sleep(2)

# Find all books
books = driver.find_elements(By.CLASS_NAME, "product_pod")

print("Total books found:", len(books))
print("-" * 50)


books_data = []

for book in books:
    title = book.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title")
    price = book.find_element(By.CLASS_NAME, "price_color").text
    availability = book.find_element(By.CLASS_NAME, "instock").text.strip()

    rating_element = book.find_element(By.CLASS_NAME, "star-rating")
    rating_classes = rating_element.get_attribute("class")
    rating = rating_classes.split()[-1]

    books_data.append({
        "title": title,
        "price": price,
        "availability": availability,
        "rating": rating
    })

print(books_data[:3])

driver.quit()

print("Collected books:", len(books_data))

<>:7: SyntaxWarning: invalid escape sequence '\A'
<>:7: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Integrated\AppData\Local\Temp\ipykernel_400\3109800664.py:7: SyntaxWarning: invalid escape sequence '\A'
  service = Service("D:\Adrta\chromedriver.exe")


Total books found: 20
--------------------------------------------------
[{'title': 'A Light in the Attic', 'price': '£51.77', 'availability': 'In stock', 'rating': 'Three'}, {'title': 'Tipping the Velvet', 'price': '£53.74', 'availability': 'In stock', 'rating': 'One'}, {'title': 'Soumission', 'price': '£50.10', 'availability': 'In stock', 'rating': 'One'}]
Collected books: 20


In [77]:
clean_text = ""

for book in books_data:
    clean_text += f"""
Title: {book['title']}
Price: {book['price']}
Availability: {book['availability']}
Rating: {book['rating']}
-------------------------
"""

print(clean_text[:1000])


Title: A Light in the Attic
Price: £51.77
Availability: In stock
Rating: Three
-------------------------

Title: Tipping the Velvet
Price: £53.74
Availability: In stock
Rating: One
-------------------------

Title: Soumission
Price: £50.10
Availability: In stock
Rating: One
-------------------------

Title: Sharp Objects
Price: £47.82
Availability: In stock
Rating: Four
-------------------------

Title: Sapiens: A Brief History of Humankind
Price: £54.23
Availability: In stock
Rating: Five
-------------------------

Title: The Requiem Red
Price: £22.65
Availability: In stock
Rating: One
-------------------------

Title: The Dirty Little Secrets of Getting Your Dream Job
Price: £33.34
Availability: In stock
Rating: Four
-------------------------

Title: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Price: £17.93
Availability: In stock
Rating: Three
-------------------------

Title: The Boys in the Boat: Nine Americans and Their Epic Quest for G

In [78]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=20
)

chunks = splitter.split_text(clean_text)

documents = [Document(page_content=chunk) for chunk in chunks]

# Embedding
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents,
    embedding_model,
    persist_directory="books_db2"
)

vectorstore.persist()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 200.60it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [79]:
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [80]:
from langchain_community.retrievers import BM25Retriever

In [81]:
all_docs = vectorstore.get()["documents"]
documents = [Document(page_content=text) for text in all_docs]

bm25 = BM25Retriever.from_documents(documents)
bm25.k = 3

In [82]:
def hybrid_retrieve(query, k_dense=3, k_bm25=3, max_chunks=6):
    
    # Dense
    dense_docs = dense_retriever.invoke(query)
    
    # Sparse
    bm25_docs = bm25.invoke(query)
    
    # Combine
    combined = dense_docs + bm25_docs
    
    # Deduplicate by content
    seen = set()
    unique_docs = []
    
    for doc in combined:
        if doc.page_content not in seen:
            unique_docs.append(doc)
            seen.add(doc.page_content)
    
    return unique_docs[:max_chunks]

In [83]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "Which books have Five rating?"

results = hybrid_retrieve(query)
context = "\n\n".join([doc.page_content for doc in results])


In [36]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a bookstore assistant.
     Answer only using the provided book data.
     If not found, say 'No matching book found.'"""),

    ("human",
     """Context:
{context}

Question:
{question}

Answer:""")
])

In [100]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query1 = "Which books have Five rating - with price?"

results = hybrid_retrieve(query1)
context = "\n\n".join([doc.page_content for doc in results])


In [101]:
context

"Title: Mesaerion: The Best Science Fiction Stories 1800-1849\nPrice: £37.59\nAvailability: In stock\nRating: One\n-------------------------\n\nTitle: Libertarianism for Beginners\nPrice: £51.33\nAvailability: In stock\nRating: Two\n-------------------------\n\nTitle: Starving Hearts (Triangular Trade Trilogy, #1)\nPrice: £13.99\nAvailability: In stock\nRating: Two\n-------------------------\n\nTitle: Shakespeare's Sonnets\nPrice: £20.66\nAvailability: In stock\nRating: Four\n-------------------------\n\nTitle: Set Me Free\nPrice: £17.46\nAvailability: In stock\nRating: Five\n-------------------------\n\nTitle: Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)\nPrice: £52.29\nAvailability: In stock\nRating: Five\n-------------------------\n\nTitle: Rip it Up and Start Again\nPrice: £35.02\nAvailability: In stock\nRating: Five\n-------------------------\n\nTitle: Sharp Objects\nPrice: £47.82\nAvailability: In stock\nRating: Four\n-------------------------\n\nTitle: Sapiens: A Brie

In [109]:
final_prompt = prompt.format(
    context=context,
    question=query1
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=final_prompt
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 2.03145007s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, '

# COINMARKETCAP - Langchain - WebBasedLoader

In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from selenium.webdriver.chrome.service import Service
service = Service("D:\Adrta\chromedriver.exe")
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options, service=service)
driver.get("https://coinmarketcap.com/")

wait = WebDriverWait(driver, 20)

# Wait until table rows appear
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody tr")))

# Scroll to load 100 rows (lazy load)
for _ in range(5):
    driver.execute_script("window.scrollBy(0, 1500);")
    time.sleep(2)

rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")

data = []

for row in rows[:100]:
    try:
        rank = row.find_element(By.CSS_SELECTOR, "td:nth-child(2)").text
        name = row.find_element(By.CSS_SELECTOR, "a[href*='/currencies/'] p").text
        price = row.find_element(By.CSS_SELECTOR, "td:nth-child(4)").text
        market_cap = row.find_element(By.CSS_SELECTOR, "td:nth-child(8)").text
        volume_24h = row.find_element(By.CSS_SELECTOR, "td:nth-child(9)").text
        
        data.append({
            "rank": rank,
            "name": name,
            "price": price,
            "market_cap": market_cap,
            "volume_24h": volume_24h
        })
    except:
        continue

driver.quit()

print(f"Total collected: {len(data)}")
print(data[:5])

<>:8: SyntaxWarning: invalid escape sequence '\A'
<>:8: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Integrated\AppData\Local\Temp\ipykernel_6296\535732417.py:8: SyntaxWarning: invalid escape sequence '\A'
  service = Service("D:\Adrta\chromedriver.exe")


Total collected: 100
[{'rank': '', 'name': 'CoinMarketCap 20 Index DTF', 'price': '$140.34', 'market_cap': '$13,695,724', 'volume_24h': '$2,881,881\n20.53K'}, {'rank': '1', 'name': 'Bitcoin', 'price': '$68,368.55', 'market_cap': '$1,365,565,727,617', 'volume_24h': '$55,035,154,854\n805.92K'}, {'rank': '2', 'name': 'Ethereum', 'price': '$2,011.74', 'market_cap': '$242,294,917,313', 'volume_24h': '$25,943,954,650\n12.92M'}, {'rank': '3', 'name': 'Tether', 'price': '$0.9998', 'market_cap': '$183,604,673,952', 'volume_24h': '$106,742,720,287\n106.75B'}, {'rank': '4', 'name': 'BNB', 'price': '$636.08', 'market_cap': '$86,735,056,772', 'volume_24h': '$1,951,718,361\n3.06M'}]


In [27]:
documents = [
    f"{d['name']} ranked {d['rank']} is priced at {d['price']}, "
    f"with market cap {d['market_cap']} and 24h volume {d['volume_24h']}."
    for d in data
]

In [28]:
documents

['CoinMarketCap 20 Index DTF ranked  is priced at $140.34, with market cap $13,695,724 and 24h volume $2,881,881\n20.53K.',
 'Bitcoin ranked 1 is priced at $68,368.55, with market cap $1,365,565,727,617 and 24h volume $55,035,154,854\n805.92K.',
 'Ethereum ranked 2 is priced at $2,011.74, with market cap $242,294,917,313 and 24h volume $25,943,954,650\n12.92M.',
 'Tether ranked 3 is priced at $0.9998, with market cap $183,604,673,952 and 24h volume $106,742,720,287\n106.75B.',
 'BNB ranked 4 is priced at $636.08, with market cap $86,735,056,772 and 24h volume $1,951,718,361\n3.06M.',
 'XRP ranked 5 is priced at $1.37, with market cap $83,797,433,359 and 24h volume $3,297,442,602\n2.40B.',
 'USDC ranked 6 is priced at $0.9999, with market cap $75,985,019,143 and 24h volume $18,301,879,897\n18.30B.',
 'Solana ranked 7 is priced at $86.24, with market cap $49,128,785,713 and 24h volume $5,371,099,008\n62.27M.',
 'TRON ranked 8 is priced at $0.2830, with market cap $26,819,141,862 and 24h 

In [17]:
options.add_argument("--disable-blink-features=AutomationControlled")

In [30]:
# chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=20,
    separators=["\n", " ", ""]
)
chunks = text_splitter.split_text("\n\n".join(documents))

In [31]:
print(len(chunks))

34


In [33]:
for i,chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print("\n")

--- Chunk 1 ---
CoinMarketCap 20 Index DTF ranked  is priced at $140.34, with market cap $13,695,724 and 24h volume $2,881,881
20.53K.

Bitcoin ranked 1 is priced at $68,368.55, with market cap $1,365,565,727,617 and 24h volume $55,035,154,854
805.92K.

Ethereum ranked 2 is priced at $2,011.74, with market cap $242,294,917,313 and 24h volume $25,943,954,650


--- Chunk 2 ---
12.92M.

Tether ranked 3 is priced at $0.9998, with market cap $183,604,673,952 and 24h volume $106,742,720,287
106.75B.

BNB ranked 4 is priced at $636.08, with market cap $86,735,056,772 and 24h volume $1,951,718,361
3.06M.

XRP ranked 5 is priced at $1.37, with market cap $83,797,433,359 and 24h volume $3,297,442,602
2.40B.


--- Chunk 3 ---
2.40B.

USDC ranked 6 is priced at $0.9999, with market cap $75,985,019,143 and 24h volume $18,301,879,897
18.30B.

Solana ranked 7 is priced at $86.24, with market cap $49,128,785,713 and 24h volume $5,371,099,008
62.27M.

TRON ranked 8 is priced at $0.2830, with market cap

In [34]:
# embedding and vectorstore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    [Document(page_content=chunk) for chunk in chunks],
    embedding_model,
    persist_directory="coin_db"
)
vectorstore.persist()

C:\Users\Integrated\AppData\Local\Temp\ipykernel_6296\3762274661.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 246.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\Integrated\AppData\Local\Temp\ipykernel_6296\3762274661.py:12: 

In [39]:
from langchain_community.retrievers import BM25Retriever

In [36]:
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [40]:
all_docs = vectorstore.get()["documents"]
documents = [Document(page_content=text) for text in all_docs]

bm25 = BM25Retriever.from_documents(documents)
bm25.k = 3

In [41]:
def hybrid_retrieve(query, k_dense=3, k_bm25=3, max_chunks=6):
    
    # Dense
    dense_docs = dense_retriever.invoke(query)
    
    # Sparse
    bm25_docs = bm25.invoke(query)
    
    # Combine
    combined = dense_docs + bm25_docs
    
    # Deduplicate by content
    seen = set()
    unique_docs = []
    
    for doc in combined:
        if doc.page_content not in seen:
            unique_docs.append(doc)
            seen.add(doc.page_content)
    
    return unique_docs[:max_chunks]

In [42]:
query = "Which coins have market cap above $50B?"
results = hybrid_retrieve(query)
context = "\n\n".join([doc.page_content for doc in results])
print(context)

CoinMarketCap 20 Index DTF ranked  is priced at $140.34, with market cap $13,695,724 and 24h volume $2,881,881
20.53K.

Bitcoin ranked 1 is priced at $68,368.55, with market cap $1,365,565,727,617 and 24h volume $55,035,154,854
805.92K.

Ethereum ranked 2 is priced at $2,011.74, with market cap $242,294,917,313 and 24h volume $25,943,954,650

326.64M.

OKB ranked 44 is priced at $76.65, with market cap $1,609,702,125 and 24h volume $19,809,055
258.45K.

Ripple USD ranked 45 is priced at $0.9999, with market cap $1,583,625,728 and 24h volume $177,133,305
177.13M.

Bitget Token ranked 46 is priced at $2.13, with market cap $1,495,235,926 and 24h volume $23,228,922
10.87M.

1.57B.

Dogecoin ranked 9 is priced at $0.09216, with market cap $15,572,621,389 and 24h volume $1,204,897,466
13.07B.

Cardano ranked 10 is priced at $0.2728, with market cap $9,844,060,446 and 24h volume $613,947,304
2.25B.

Bitcoin Cash ranked 11 is priced at $443.97, with market cap $8,880,315,765 and 24h volume $4

In [44]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"""You are a crypto assistant.
Answer only using the provided coin data.
If not found, say 'No matching coin found.'
Context:
{context}
Question:
{query}
Answer:"""
)
print(response.text)

Bitcoin
Ethereum
Tether
BNB
XRP


In [47]:
def answer_query(query):
    results = hybrid_retrieve(query)
    context = "\n\n".join([doc.page_content for doc in results])
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=f"""You are a crypto assistant.
Answer only using the provided coin data.
If not found, say 'No matching coin found.'
Context:
{context}
Question:
{query}
Answer:"""
    )
    return response.text

In [48]:
query = "Which coins are in top 5 by market cap?"

answer = answer_query(query)
print(answer)


1. Bitcoin
2. Ethereum
3. Tether
4. BNB
5. XRP
